In [1]:
!pip install groq python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.6 MB/s eta 0:00:00


In [5]:
import os
import getpass
from groq import Groq

# Ask user to manually enter API key
groq_key = getpass.getpass("Enter your GROQ API Key: ")

# Set environment variable
os.environ["GROQ_API_KEY"] = groq_key

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("✅ Groq client initialized successfully!")


Enter your GROQ API Key: ··········
✅ Groq client initialized successfully!


In [22]:
MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Senior Technical Support Engineer.
Be precise, code-focused, and analytical.
Provide debugging steps and code examples when necessary.
"""
    },
    "billing": {
        "system_prompt": """
You are a Billing Support Specialist.
Be empathetic, professional, and policy-driven.
Clearly explain refund and subscription policies.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly Customer Support Assistant.
Handle casual queries politely and clearly.
"""
    }
}


In [23]:
def route_prompt(user_input):
    routing_prompt = f"""
Classify the following text into one of these categories:
[technical, billing, general, tool]

If the user is asking about live data like cryptocurrency price,
return "tool".

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    return response.choices[0].message.content.strip().lower()


In [24]:
def get_bitcoin_price():
    return "The current price of Bitcoin is $62,450 (mock data)."


In [25]:
def process_request(user_input):
    category = route_prompt(user_input)

    print("🔎 Routed to:", category)

    if category == "tool":
        return get_bitcoin_price()

    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content


In [26]:
print(process_request("My python script is throwing an IndexError on line 5."))


🔎 Routed to: technical
I'd be happy to help you debug your Python script. 

To assist you better, I'll need some more information:

1. **The code**: Please share the relevant code snippet where the error is occurring. The more code you share, the better I can understand the context.

2. **The error message**: The full error message, including the line number and any additional details.

3. **The input/data**: If the error is related to user input or external data, please provide an example of the input/data that causes the error.

Here are some general steps you can follow to troubleshoot the issue:

1. **Check the line number**: Verify that the line number mentioned in the error message matches the line of code in your script.

2. **Review the indexing**: Ensure that you're not trying to access an index that is out of range. Python list indices start at 0, so if you're trying to access an element at index `n`, it must be less than the length of the list.

3. **Inspect the data**: If t

In [27]:
print(process_request("I was charged twice for my subscription this month."))


🔎 Routed to: billing
I'm so sorry to hear that you've been charged twice for your subscription. I'm here to help you resolve this issue as quickly and efficiently as possible.

To better assist you, could you please provide me with some details about your subscription, such as:

1. Your account name (username or email address associated with your account)
2. The date and amount of the duplicate charge
3. The subscription plan you're currently on

Once I have this information, I'll be happy to look into the matter further and provide you with a refund for the duplicate charge.

Additionally, I'd like to let you know that our refund policy states that we'll refund any duplicate charges to the original payment method within 3-5 business days after the issue has been resolved. Please note that this timeframe may vary depending on your bank's processing times.

If you're experiencing any further issues, please don't hesitate to reach out to me directly. I'm here to help you get back on trac

In [28]:
print(process_request("What is the current price of Bitcoin?"))

🔎 Routed to: tool
The current price of Bitcoin is $62,450 (mock data).
